# 🎰 The slot machine game

You are in a casino with **4 slot machines: A, B, C and D**. Each machine pays out a random amount every time you play it,
but some machines pay **more on average** than others. Nobody tells you which ones.

You get **25 plays**. Your goal: **earn as much money as possible in total.**

### How to play
1. Fill in your **name** and the **class code** your teacher gives you in the cell below.
2. Run the cell (click ▶ or press Shift+Enter). Buttons appear.
3. Click a machine to play it. You see what it paid. Think about your strategy!
4. After 25 plays, you get a **result code**. Submit it as your teacher explains.

**One game only, no restarts**: rerunning the cell starts a new game, but that's not playing fair. 🙂

In [ ]:
#@title ▶ Step 1: fill in your name and the class code, then run this cell { display-mode: "form" }
NAME = "WV" #@param {type:"string"}
CLASS_CODE = "DSFB2026" #@param {type:"string"}

# ------------------------------------------------------------------ (no need to read below)

import hashlib, re, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*vert.*")
import numpy as np

ROUNDS = 25          # rounds per game (must be the same in student and teacher notebook)
LABELS = "ABCD"
SD = 2.5             # noise on payouts (euro)

def clean_code(code):
    return re.sub(r"[^A-Za-z0-9]", "", str(code)).upper() or "DEFAULT"

def make_machines(class_code, rounds=ROUNDS):
    """Hidden machines, fully determined by the class code.
    Returns the true average payout per machine and, per machine, the sequence of payouts
    for its 1st, 2nd, 3rd, ... play. Everyone with the same class code faces the same machines and the same luck."""
    seed = int(hashlib.sha256(clean_code(class_code).encode()).hexdigest(), 16) % 2**32
    rng = np.random.default_rng(seed)
    base = np.array([4.0, 5.0, 6.0, 7.5])
    means = np.round(rng.permutation(base) + rng.uniform(-0.4, 0.4, 4), 1)
    seqs = np.clip(np.round(rng.normal(means[:, None], SD, (4, rounds))), 0, None).astype(int)
    return means, seqs

def score(choices, seqs):
    """Total payout of a sequence of choices like 'ABBCC...'."""
    counts, total = [0, 0, 0, 0], 0
    for ch in choices:
        j = LABELS.index(ch)
        total += int(seqs[j][counts[j]]); counts[j] += 1
    return total

# ---- simple strategies, used to compare students with algorithms ----
def play_strategy(name, means, rng, rounds=ROUNDS):
    k = len(means); n = np.zeros(k); s = np.zeros(k); total = 0; picks = []
    for t in range(rounds):
        if name == "Random":
            j = rng.integers(k)
        elif name == "Explore-then-commit (3 each)":
            j = t % k if t < 3 * k else int(np.argmax(s / np.maximum(n, 1)))
        elif name == "Epsilon-greedy (10%)":
            if t < k: j = t
            elif rng.random() < 0.1: j = rng.integers(k)
            else: j = int(np.argmax(s / np.maximum(n, 1)))
        elif name == "UCB (optimism)":
            if t < k: j = t                                   # play each machine once first
            else:                                              # average + uncertainty bonus (scaled to the payout noise)
                j = int(np.argmax(s / n + SD * np.sqrt(2 * np.log(t + 1) / n)))
        elif name == "Thompson sampling":
            prior_m, prior_v, v = 5.0, 9.0, SD ** 2          # normal beliefs about each machine's average
            post_v = 1 / (1 / prior_v + n / v)
            post_m = post_v * (prior_m / prior_v + s / v)
            j = int(np.argmax(rng.normal(post_m, np.sqrt(post_v))))
        elif name == "Perfect knowledge":
            j = int(np.argmax(means))
        r = max(0, round(rng.normal(means[j], SD)))
        n[j] += 1; s[j] += r; total += r; picks.append(j)
    return total, picks

STRATEGIES = ["Random", "Explore-then-commit (3 each)", "Epsilon-greedy (10%)", "UCB (optimism)", "Thompson sampling", "Perfect knowledge"]

import ipywidgets as w
import matplotlib.pyplot as plt
from IPython.display import display, HTML
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

import unicodedata
NAME_CLEAN = re.sub(r"[^A-Za-z0-9]+", "_", unicodedata.normalize("NFKD", NAME).encode("ascii", "ignore").decode().strip()).strip("_") or "anonymous"
CODE = clean_code(CLASS_CODE)
_MEANS, _SEQS = make_machines(CODE)
COLORS = ["#466E87", "#E7B037", "#1D8DB0", "#C00000"]

class Game:
    def __init__(self):
        self.choices, self.payouts = [], []
        self.counts = [0, 0, 0, 0]
        self.history = [[] for _ in range(4)]

    @property
    def done(self): return len(self.choices) >= ROUNDS
    @property
    def total(self): return sum(self.payouts)

    def play(self, j):
        if self.done: return None
        p = int(_SEQS[j][self.counts[j]])
        self.counts[j] += 1
        self.choices.append(LABELS[j]); self.payouts.append(p); self.history[j].append(p)
        return p

    def result_code(self):
        return f"MAB-{CODE}-{NAME_CLEAN}-{''.join(self.choices)}"

GAME = Game()

# ------------------------------------------------------------------ user interface
status = w.HTML()
message = w.HTML()
table = w.HTML()
final = w.Output()
buttons = [w.Button(description=f"Machine {L}", layout=w.Layout(width="150px", height="60px"),
                    style={"button_color": c})
           for L, c in zip(LABELS, ["#A9C0CE", "#F2D38F", "#9BD0E3", "#F2A7A7"])]

def render():
    status.value = (f"<h3 style='margin:4px 0'>Round {min(len(GAME.choices) + 1, ROUNDS)} of {ROUNDS} · "
                    f"Total earned: <span style='color:#1D8DB0'>€{GAME.total}</span></h3>")
    rows = ""
    for j, L in enumerate(LABELS):
        h = GAME.history[j]
        avg = f"€{np.mean(h):.1f}" if h else "–"
        last = ", ".join(f"€{x}" for x in h[-6:]) if h else "–"
        rows += (f"<tr><td style='padding:4px 12px;color:{COLORS[j]}'><b>Machine {L}</b></td>"
                 f"<td style='padding:4px 12px;text-align:center'>{len(h)}</td>"
                 f"<td style='padding:4px 12px;text-align:center'>€{sum(h)}</td>"
                 f"<td style='padding:4px 12px;text-align:center'>{avg}</td>"
                 f"<td style='padding:4px 12px'>{last}</td></tr>")
    table.value = ("<table style='border-collapse:collapse;font-size:15px'>"
                   "<tr style='border-bottom:1px solid #999'><th style='padding:4px 12px'>Machine</th><th>Times played</th>"
                   "<th>Total</th><th>Average</th><th style='padding:4px 12px;text-align:left'>Last payouts</th></tr>"
                   + rows + "</table>")

def on_click(b):
    j = buttons.index(b)
    p = GAME.play(j)
    if p is None: return
    message.value = f"<p style='font-size:16px'>Machine {LABELS[j]} paid <b>€{p}</b>.</p>"
    render()
    if GAME.done: finish()

def finish():
    for b in buttons: b.disabled = True
    status.value = f"<h3 style='margin:4px 0'>Game over! You earned <span style='color:#1D8DB0'>€{GAME.total}</span> in {ROUNDS} plays.</h3>"
    with final:
        display(HTML(
            "<div style='border:2px solid #1D8DB0;border-radius:8px;padding:10px;margin:8px 0;max-width:720px'>"
            "<b>Your result code</b> (copy it completely and submit it as your teacher explains):<br>"
            f"<code style='font-size:16px;user-select:all'>{GAME.result_code()}</code></div>"
            "<p>Before you look further: <b>write down in one sentence how you decided which machine to play.</b></p>"
            "<p>Then run <b>Step 2</b> below to see how algorithms would have done on the same machines.</p>"))
        fig, ax = plt.subplots(figsize=(9, 2.8))
        for j in range(4):
            idx = [t for t, c in enumerate(GAME.choices) if c == LABELS[j]]
            ax.scatter([t + 1 for t in idx], [GAME.payouts[t] for t in idx], s=70, color=COLORS[j], label=f"Machine {LABELS[j]}")
        ax.set_xlabel("round"); ax.set_ylabel("payout (€)"); ax.set_title("Your game: which machine you played, and what it paid")
        ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.3), frameon=False)
        for s_ in ["top", "right"]: ax.spines[s_].set_visible(False)
        plt.tight_layout(); plt.show()

for b in buttons: b.on_click(on_click)
render()
if not NAME.strip() or not CLASS_CODE.strip():
    display(HTML("<p style='color:#C00000;font-size:16px'><b>Please fill in your name and the class code first, then run this cell again.</b></p>"))
else:
    display(w.VBox([status, w.HBox(buttons), message, table, final]))

### Step 2: after the game, compare yourself with algorithms
Only run this **after** you finished your 25 plays (and wrote down your strategy).

In [ ]:
#@title ▶ Step 2: reveal the machines and compare with algorithms { display-mode: "form" }
if not GAME.done:
    print("Finish your game first! 🙂")
else:
    best = int(np.argmax(_MEANS))
    print("The machines' true average payouts were:")
    for j, L in enumerate(LABELS):
        print(f"  Machine {L}: €{_MEANS[j]:.1f}" + ("   ← the best machine" if j == best else ""), f"(you played it {GAME.counts[j]}×)")
    print(f"\nYou earned €{GAME.total}. Always playing the best machine earns about €{ROUNDS * _MEANS[best]:.0f} on average.")
    rng_s = np.random.default_rng(0)
    res = {s: [play_strategy(s, _MEANS, rng_s)[0] for _ in range(2000)] for s in STRATEGIES}
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.boxplot(list(res.values()), vert=False, showfliers=False, widths=0.6)
    ax.set_yticks(range(1, len(res) + 1)); ax.set_yticklabels(list(res.keys()))
    ax.axvline(GAME.total, color="#C00000", lw=3, label=f"You: €{GAME.total}")
    ax.set_xlabel("total earned in 25 plays (€), 2,000 simulated games per algorithm")
    ax.set_title("How you compare with algorithms on the same machines"); ax.legend(loc="lower right")
    for s_ in ["top", "right"]: ax.spines[s_].set_visible(False)
    plt.tight_layout(); plt.show()
    for s in STRATEGIES[:-1]:
        print(f"You beat {np.mean(np.array(res[s]) < GAME.total):.0%} of the games played by: {s}")